In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/variables

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
etl_input_table_validator(
    silver_transaction_fiscal_header, silver_fiscal_days, 
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)

In [0]:
header_fiscal = spark.table(silver_transaction_fiscal_header)
fwe_min = header_fiscal.agg({"FISCAL_WEEK_END": "min"}).collect()[0][0]
fwe_max = header_fiscal.agg({"FISCAL_WEEK_END": "max"}).collect()[0][0]

members = header_fiscal.select("MBRSHP_SID").distinct()

fiscal_days = spark.table(silver_fiscal_days)
fiscal_weeks = fiscal_days.drop("FISCAL_DAY").distinct()
fiscal_weeks_inscope = fiscal_weeks.filter(
    'FISCAL_WEEK_END between "{}" and "{}"'.format(fwe_min, fwe_max)
)

df_skeleton = members.crossJoin(fiscal_weeks_inscope)

df_skeleton.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'skeleton', config_validation, df_skeleton, stats_etl_path,  [validations.TestColNames, validations.TestDuplicates],
    )

### Merge

In [0]:
df_skeleton.write.mode("overwrite").saveAsTable(silver_skeleton)

if archive_flag:
    save_archive(df_skeleton, silver_skeleton_archive, run_as_date)